# Chronos with a 2048-hour history

The earlier run gave Chronos only 336 hours (two weeks) of history. Chronos-Bolt
normally reads up to 2048 hours (about 12 weeks). This notebook repeats the same
60 forecast cases with the longer history. Nothing else changes: same model
commit, same test hours, same baselines.

Q1 starts in February, so its history is shorter than 2048 hours; it uses all
available history. Missing hours are passed as gaps, which Chronos ignores.

**Run:** Runtime > Change runtime type > T4 GPU. Run the cells in order.
Upload `long_context_chronos_inputs.zip` when asked. At the end, download
`long_context_chronos_results.zip` and put it in the project folder.

In [ ]:
import os, subprocess, sys
from pathlib import Path

ENV = Path("/content/chronos_forecast_env")
PYTHON = str(ENV / "bin/python")
BOOTSTRAP = Path("/content/chronos_env_bootstrap")
# Isolated environment, so Colab's own packages cannot conflict.
RUN_ENV = {key: value for key, value in os.environ.items()
           if key not in ("PYTHONPATH", "PYTHONHOME")}
RUN_ENV["PYTHONNOUSERSITE"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "--target", str(BOOTSTRAP), "virtualenv"], env=RUN_ENV, check=True)
subprocess.run([sys.executable, "-m", "virtualenv", str(ENV)],
               env={**RUN_ENV, "PYTHONPATH": str(BOOTSTRAP)}, check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--quiet",
                "torch==2.11.0", "chronos-forecasting==2.2.2",
                "transformers==4.57.6", "numpy==2.1.3", "pandas==2.2.3"],
               env=RUN_ENV, check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], env=RUN_ENV, check=True)
print("Setup complete")

In [ ]:
subprocess.run([PYTHON, "-c", "import torch; from chronos import BaseChronosPipeline; "
                "assert torch.cuda.is_available(), 'Select a GPU runtime first'; "
                "print(torch.__version__, torch.cuda.get_device_name(0))"],
               env=RUN_ENV, check=True)

In [ ]:
from google.colab import files
from pathlib import PurePosixPath
import hashlib, zipfile

ROOT = Path("/content/long_context_chronos")
ROOT.mkdir(exist_ok=True)
uploaded = files.upload()
assert len(uploaded) == 1, "Choose only long_context_chronos_inputs.zip"
filename, payload = next(iter(uploaded.items()))
assert hashlib.sha256(payload).hexdigest() == "cf36dc20f0990adb93cb24a41247a403fa7149dfb01582a22712d88ae3813dc6", "Wrong or changed ZIP"
with zipfile.ZipFile(filename) as archive:
    for member in archive.infolist():
        parts = PurePosixPath(member.filename)
        assert not parts.is_absolute() and ".." not in parts.parts and "\\" not in member.filename
        assert (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
del uploaded, payload
RUNNER = str(ROOT / "chronos_long_context.py")
for label in ("Q1", "Q2", "Q3", "Q4"):
    subprocess.run([PYTHON, RUNNER, "--root", str(ROOT / label), "--check-only"], env=RUN_ENV, check=True)

## Run all four periods

60 cases. Finished cases are saved under `results/Q1` etc. If the run stops,
run this cell again: finished cases are reused. Downloading partial results
before disconnecting keeps them.

In [ ]:
OUTPUT = ROOT / "results"
for label in ("Q1", "Q2", "Q3", "Q4"):
    subprocess.run([PYTHON, "-u", RUNNER, "--root", str(ROOT / label),
                    "--output", str(OUTPUT / label), "--batch-size", "32"], env=RUN_ENV, check=True)

In [ ]:
import csv
print("SEVEN-DAY RESULTS, 2048-hour history (positive = Chronos has lower error)")
for label in ("Q1", "Q2", "Q3", "Q4"):
    assert (OUTPUT / label / "complete.json").exists(), f"Incomplete window: {label}"
    with (OUTPUT / label / "comparison.csv").open(newline="") as stream:
        for row in csv.DictReader(stream):
            if row["baseline"] == "selected" and row["horizon_hours"] == "168":
                gain = row["chronos_MAE_gain_pct"].strip()
                gain = f"{float(gain):+.2f}%" if gain else "undefined (baseline MAE is zero)"
                print(f"{label} | {row['dataset']} | {row['series']} | {row['baseline_rule']} | "
                      f"Chronos MAE {float(row['chronos_MAE']):.6g} | gain {gain}")
print("Descriptive only. The uncertainty check runs locally.")

In [ ]:
import shutil
result_zip = shutil.make_archive("/content/long_context_chronos_results", "zip", OUTPUT)
files.download(result_zip)